<a href="https://colab.research.google.com/github/shivamkumar359/03-rd-SEM--ASSIGNMENTS/blob/main/ML_W_10_ML_ASSIGNMENT_W_10_Q_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# TASK 1: ANALYZE MODEL PERFORMANCE AND BIAS-VARIANCE TRADEOFF
# IIT Jodhpur BSc - Machine Learning
# ============================================================

import os
import sys
import zipfile
import subprocess

In [3]:
# ============================================================
# 1. CREATE THE PYTHON SOURCE FILE
# ============================================================


import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline



In [7]:
# ------------------------------------------------------------
# 1. DATA GENERATION
# ------------------------------------------------------------

np.random.seed(42)

def true_function(x):
    return np.sin(1.5 * np.pi * x)

noise_std = 0.20


# Training dataset
X_train = np.random.uniform(-1, 1, 80).reshape(-1, 1)
y_train = true_function(X_train.ravel()) + np.random.normal(
    0, noise_std, 80
)


# Validation dataset
X_val = np.linspace(-1, 1, 300).reshape(-1, 1)
y_val = true_function(X_val.ravel()) + np.random.normal(
    0, noise_std, 300
)


# Model complexity levels
degrees = range(1, 16)

training_errors = []
validation_errors = []
bias_squared_values = []
variance_values = []
irreducible_errors = []
total_errors = []

In [8]:
# ------------------------------------------------------------
# 2. TRAINING AND VALIDATION ERROR CALCULATION
# ------------------------------------------------------------

for degree in degrees:

    model = make_pipeline(
        PolynomialFeatures(degree=degree),
        LinearRegression()
    )

    model.fit(X_train, y_train)

    train_predictions = model.predict(X_train)
    val_predictions = model.predict(X_val)

    train_mse = np.mean((y_train - train_predictions) ** 2)
    val_mse = np.mean((y_val - val_predictions) ** 2)

    training_errors.append(train_mse)
    validation_errors.append(val_mse)

    # --------------------------------------------------------
    # 3. BIAS-VARIANCE DECOMPOSITION
    # --------------------------------------------------------

    ensemble_predictions = []

    # Train multiple models using independent datasets
    for _ in range(60):

        X_sample = np.random.uniform(-1, 1, 80).reshape(-1, 1)

        y_sample = (
            true_function(X_sample.ravel())
            + np.random.normal(0, noise_std, 80)
        )

        sample_model = make_pipeline(
            PolynomialFeatures(degree=degree),
            LinearRegression()
        )

        sample_model.fit(X_sample, y_sample)

        predictions = sample_model.predict(X_val)
        ensemble_predictions.append(predictions)

    ensemble_predictions = np.array(ensemble_predictions)

    mean_predictions = np.mean(ensemble_predictions, axis=0)

    # Bias squared
    bias_squared = np.mean(
        (mean_predictions - true_function(X_val.ravel())) ** 2
    )

    # Variance
    variance = np.mean(
        np.var(ensemble_predictions, axis=0)
    )

    # Irreducible error (known noise variance)
    irreducible_error = noise_std ** 2

    # Total prediction error
    total_error = (
        bias_squared + variance + irreducible_error
    )

    bias_squared_values.append(bias_squared)
    variance_values.append(variance)
    irreducible_errors.append(irreducible_error)
    total_errors.append(total_error)


In [9]:
# ------------------------------------------------------------
# 4. IDENTIFY OPTIMAL MODEL COMPLEXITY
# ------------------------------------------------------------

optimal_index = np.argmin(validation_errors)
optimal_degree = list(degrees)[optimal_index]

minimum_validation_error = validation_errors[optimal_index]

optimal_bias = bias_squared_values[optimal_index]
optimal_variance = variance_values[optimal_index]
optimal_noise = irreducible_errors[optimal_index]
optimal_total = total_errors[optimal_index]




In [11]:
# ------------------------------------------------------------
# 5. IDENTIFY UNDERFITTING AND OVERFITTING
# ------------------------------------------------------------

underfitting_degrees = [
    d for d, train_error, val_error in zip(
        degrees, training_errors, validation_errors
    )
    if d < optimal_degree
    and train_error > minimum_validation_error * 1.5
]

overfitting_degrees = [
    d for d, train_error, val_error in zip(
        degrees, training_errors, validation_errors
    )
    if d > optimal_degree
    and val_error > minimum_validation_error * 1.2
    and val_error > train_error
]